# Individual and pooled diagnostic performance

Figure 3 and Table 2. Run notebook 01 first.

## Step 1: Load the scores

In [ ]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'syllableseparation_apple_corrected.csv').is_file():
    raise FileNotFoundError('Open this notebook from the repository folder.')
output_dir = ROOT / 'results'
output_dir.mkdir(exist_ok=True)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9,
                     'axes.linewidth': .8, 'pdf.fonttype': 42})

def save_figure(fig, name):
    fig.savefig(output_dir / f'{name}.png', dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(output_dir / f'{name}.pdf', bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
df = pd.read_csv(ROOT / 'Syllableerror_scores_raw_and_normalized.csv', dtype={'sub_id': str})
features = [f'error_rate_{i}' for i in range(1, 5)]
raw_columns = [f'error_{i}' for i in range(1, 5)]
sentences = ['LBP', 'LAP', 'VP', 'MP']
assert len(df) == 74 and df['sub_id'].is_unique
assert df['disease'].value_counts().to_dict() == {0: 37, 1: 37}
assert np.isfinite(df[features + raw_columns].to_numpy()).all()
assert np.allclose(df[features], df[raw_columns].to_numpy() / [8, 8, 7, 11])
df['pooled'] = df[raw_columns].sum(axis=1) / 34

## Step 2: Define paired ROC calculations

In [ ]:
from scipy.stats import norm
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from statsmodels.stats.multitest import multipletests

def delong(labels, scores):
    labels = np.asarray(labels)
    scores = np.asarray(scores, dtype=float)
    if scores.ndim == 1:
        scores = scores[:, None]
    positive, negative = scores[labels == 1].T, scores[labels == 0].T
    h = (positive[:, :, None] > negative[:, None, :]).astype(float)
    h += .5 * (positive[:, :, None] == negative[:, None, :])
    covariance = (np.atleast_2d(np.cov(h.mean(axis=2), ddof=1)) / positive.shape[1]
                  + np.atleast_2d(np.cov(h.mean(axis=1), ddof=1)) / negative.shape[1])
    return h.mean(axis=(1, 2)), covariance

def auc_table(labels, scores, names):
    auc, covariance = delong(labels, scores)
    se = np.sqrt(np.maximum(0, np.diag(covariance)))
    table = pd.DataFrame({'Sentence': names, 'AUC': auc,
                          'Lower95': np.maximum(0, auc - norm.ppf(.975) * se),
                          'Upper95': np.minimum(1, auc + norm.ppf(.975) * se)})
    assert np.allclose(auc, [roc_auc_score(labels, s) for s in np.asarray(scores).T])
    return table, covariance

def compare_auc(table, covariance, pairs):
    rows = []
    for i, j in pairs:
        difference = table.iloc[i]['AUC'] - table.iloc[j]['AUC']
        se = np.sqrt(max(0, covariance[i, i] + covariance[j, j] - 2 * covariance[i, j]))
        p = 2 * norm.sf(abs(difference) / se) if se > 0 else (1. if abs(difference) < 1e-12 else np.nan)
        rows.append({'A': table.iloc[i]['Sentence'], 'B': table.iloc[j]['Sentence'],
                     'AUC_difference': difference, 'Lower95': max(-1, difference - norm.ppf(.975) * se),
                     'Upper95': min(1, difference + norm.ppf(.975) * se), 'p_raw': p})
    result = pd.DataFrame(rows)
    if not np.isfinite(result['p_raw']).all():
        raise ValueError('Undefined DeLong comparison.')
    result['p_Holm'] = multipletests(result['p_raw'], method='holm')[1]
    return result

def youden(labels, scores):
    fpr, tpr, thresholds = roc_curve(labels, scores, drop_intermediate=False)
    valid = np.isfinite(thresholds)
    thresholds, j = thresholds[valid], (tpr - fpr)[valid]
    return float(thresholds[np.isclose(j, j.max(), rtol=0, atol=1e-12)].max())

def ratio(a, b):
    return a / b if b else (np.inf if a else np.nan)

def performance(labels, scores, threshold):
    tn, fp, fn, tp = confusion_matrix(labels, np.asarray(scores) >= threshold, labels=[0, 1]).ravel()
    sensitivity, specificity = ratio(tp, tp + fn), ratio(tn, tn + fp)
    return {'Threshold': threshold, 'TP': int(tp), 'FN': int(fn), 'TN': int(tn), 'FP': int(fp),
            'Accuracy': (tp + tn) / len(labels), 'Sensitivity': sensitivity, 'Specificity': specificity,
            'Youden_J': sensitivity + specificity - 1,
            'LR+': ratio(sensitivity, 1 - specificity), 'LR-': ratio(1 - sensitivity, specificity)}

## Step 3: Compare AUCs

In [ ]:
from itertools import combinations
names = sentences + ['Pooled']
y = df['disease'].to_numpy()
scores = df[features + ['pooled']].to_numpy()
auc_df, covariance = auc_table(y, scores, names)
comparison_df = compare_auc(auc_df, covariance, list(combinations(range(5), 2)))
auc_df.to_csv(output_dir / 'Figure3_AUC.csv', index=False)
comparison_df.to_csv(output_dir / 'Figure3_paired_DeLong_Holm10.csv', index=False)
display(auc_df.round(3))
display(comparison_df.round(6))

## Step 4: Calculate Table 2

In [ ]:
rows = []
for i, name in enumerate(names):
    threshold = youden(y, scores[:, i])
    rows.append({'Sentence': name, **performance(y, scores[:, i], threshold)})
performance_df = pd.DataFrame(rows).merge(auc_df, on='Sentence')
performance_df.to_csv(output_dir / 'Table2_diagnostic_performance.csv', index=False)
display(performance_df.round(3))

## Step 5: Draw Figure 3

In [ ]:
colors = {'LBP': '#0072B2', 'LAP': '#D55E00', 'VP': '#009E73',
          'MP': '#CC79A7', 'Pooled': '#222222', 'Combined': '#7A5195'}

def auc_bars(ax, table, comparisons, title):
    names = table['Sentence'].tolist()
    auc = table['AUC'].to_numpy()
    lower, upper = table['Lower95'].to_numpy(), table['Upper95'].to_numpy()
    x = np.arange(len(names))
    ax.bar(x, auc, width=.62, color=[colors[n] for n in names], zorder=3)
    ax.errorbar(x, auc, yerr=[auc - lower, upper - auc], fmt='none',
                ecolor='.25', elinewidth=1, capsize=4, zorder=4)
    ax.axhline(.5, color='.6', linestyle='--', linewidth=.8)
    significant = comparisons.loc[comparisons['p_Holm'] < .05].copy()
    significant['span'] = [abs(names.index(a) - names.index(b)) for a, b in zip(significant.A, significant.B)]
    significant = significant.sort_values('span', kind='stable').reset_index(drop=True)
    for level, row in significant.iterrows():
        left, right = sorted([names.index(row.A), names.index(row.B)])
        height = 1.10 + level * .075
        ax.plot([left, left, right, right], [height, height + .018, height + .018, height], color='.3', lw=.8)
        stars = '***' if row.p_Holm < .001 else '**' if row.p_Holm < .01 else '*'
        ax.text((left + right) / 2, height + .021, stars, ha='center', va='bottom')
    top = 1.24 + max(0, len(significant) - 1) * .075
    ax.set(xlim=(-.6, len(names) - .4), ylim=(0, top), xticks=x, xticklabels=names,
           yticks=np.arange(0, 1.01, .2), ylabel='AUC')
    ax.set_title(title, loc='left', fontweight='bold', fontsize=10, pad=10)
    ax.set_axisbelow(True)
    ax.grid(axis='y', color='.9', linewidth=.5)
    table_artist = ax.table(cellText=[[f'{a:.3f}' for a in auc],
                                    [f'{l:.3f}–{u:.3f}' for l, u in zip(lower, upper)]],
                            rowLabels=['AUC', '95% CI'], cellLoc='center', rowLoc='right',
                            bbox=[0, -.28, 1, .15])
    table_artist.auto_set_font_size(False)
    table_artist.set_fontsize(7)
    for (row, col), cell in table_artist.get_celld().items():
        cell.set_edgecolor('white')
        if row == 0 and col >= 0:
            cell.get_text().set_color(colors[names[col]])
            cell.get_text().set_fontweight('bold')
fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.7), gridspec_kw={'width_ratios': [1, 1.5]})
ax = axes[0]
ax.plot([0, 1], [0, 1], color='.65', linestyle=':', lw=.8)
for i, name in enumerate(names):
    fpr, tpr, _ = roc_curve(y, scores[:, i])
    row = performance_df.iloc[i]
    ax.plot(fpr, tpr, color=colors[name], linestyle='--' if name == 'Pooled' else '-',
            label=f'{name}: {row.AUC:.3f}', lw=1.4)
    ax.scatter(1-row.Specificity, row.Sensitivity, facecolors='white', edgecolors=colors[name], s=25, zorder=4)
ax.set(xlim=(-.02, 1.02), ylim=(-.02, 1.02), xlabel='1 − specificity', ylabel='Sensitivity')
ax.set_aspect('equal')
ax.set_title('a  ROC curves', loc='left', fontsize=10, fontweight='bold')
ax.legend(loc='lower right', frameon=False, fontsize=8)
auc_bars(axes[1], auc_df, comparison_df, 'b  AUC comparisons')
fig.subplots_adjust(left=.07, right=.99, bottom=.27, top=.90, wspace=.35)
save_figure(fig, 'Figure3_ROC_AUC')